# Bronze Layer: Sensor Reading Ingestion

**Purpose:** Ingest IoT sensor readings with data quality validation

**Input:** `/Volumns/dev/bronze/sample_data/sensor_readings_10k.csv`

**Output:** 
- `dev.bronze.sensor_readings_raw` (all data)
- `dev.bronze.sensor_readings_quarantine` (bad data)

**Data Quality Issue:**
- ~5% missing values
- ~3% invalid data types
- ~2% outliners
- ~2% duplicates

#Configuration

In [0]:
# Configuration
CATALOG = 'dev'
SCHEMA = 'bronze'
TABLE_RAW = 'sensor_readings_raw'
TABLE_QUARANTINE = 'sensor_readings_quarantine'

TABLE_RAW_FULL = f"{CATALOG}.{SCHEMA}.{TABLE_RAW}"
TABLE_QUARANTINE_FULL = f"{CATALOG}.{SCHEMA}.{TABLE_QUARANTINE}"

print("Target Table: ")
print(f"Main Table: {TABLE_RAW_FULL}")
print(f"Quarantine: {TABLE_QUARANTINE_FULL}")

## Define Schema
Explicitly define schema with proper data types


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

#Defind schema for sensor readings
sensor_schema = StructType([
    StructField("equipment_id", StringType(), False),
    StructField("sensor_type", StringType(), False),
    StructField("timestamp", StringType(), True),
    StructField("value", StringType(), True),
    StructField("unit", StringType(), True),
    StructField("sensor_id", StringType(), True),
    StructField("data_source", StringType(), True),
    StructField("quality_flag", StringType(), True)
])

print("Schema Defind")
print(f"Columns: {[f.name for f in sensor_schema.fields]}")

## Read Raw CSV
Load data without validation - accept everything initially 

In [0]:
raw_df = spark.read.csv("/Volumes/dev/bronze/sample_data/sensor_readings_10k.csv", header=True, schema=sensor_schema)

print(f"Total raw count: {raw_df.count()}")
print("First few records: ")
raw_df.show(5, truncate=False)

## Add Ingestion Metadata

add audit columns to track when and how data was loaded

In [0]:
from pyspark.sql.functions import current_timestamp, lit, input_file_name

# Add metadata columns
enriched_df = raw_df \
                .withColumn("ingestion_timestamp", current_timestamp()) \
                .withColumn("source_file", lit("sensor_readings_10k.csv")) \
                .withColumn("pipline_name", lit("bronze_sensor_ingestion")) \
                .withColumn("pipline_version", lit("1.0"))

print("Metadata columns added")
enriched_df.select("equipment_id", "sensor_type", "ingestion_timestamp", "quality_flag").show(5)

## Data Qualtiy Analysis
Analyze the quality flags that comes with the data

In [0]:
from pyspark.sql.functions import col, count

# Count by qualtiy flag
print("Data Quality Summary")
quality_summary = enriched_df.groupBy("quality_flag").count().orderBy("count", ascending=False)
quality_summary.show()

# Calculate percentages
total_records = enriched_df.count()
print(f"Total recods: {total_records}")

for row in quality_summary.collect():
    flag = row['quality_flag']
    cnt = row['count']
    pct = (cnt / total_records) * 100
    print(f"{flag} : {cnt} ({pct:.1f}%)")

## Separate Good Data from Bad Data

Split into two dataframes: valid records and quarantine records

In [0]:
# Good data - only GOOD quality flag
good_df = enriched_df.filter(col("quality_flag") == 'GOOD')

# Quarantine data - everything else
bad_df = enriched_df.filter(col("quality_flag") != "GOOD")

print(f"Good Records: {good_df.count()}")
print(f"Bad records (quarantine): {bad_df.count()}")

# Show sample of Bad data
print("\nSample quarantine records")
bad_df.select("equipment_id", "sensor_type", "value", "quality_flag").show(10, truncate=False)

## Type Coversation and Validation 
convert string column to proper types for good data

In [0]:
from pyspark.sql.functions import to_timestamp, col

# convert timestamp from string to timestamp type
# convert value from string to double
validated_df = good_df \
        .withColumn("timestamp_parsed", to_timestamp(col("timestamp"))) \
        .withColumn("value_parsed", col("value").cast(DoubleType())) \
        .drop('timestamp', 'value') \
        .withColumnRenamed("timestamp_parsed", "timestamp") \
        .withColumnRenamed("value_parsed", "value")
    
print("Type coversion complated")
print("\nSchema after conversion")
validated_df.printSchema()

# verify no nulls in critical columns after conversion
null_check = validated_df.filter(
    col("timestamp").isNull() | col("value").isNull()
).count()

print(f"\nNull values after conversion: {null_check} (should be 0 for good Data)")


## Write to Bronze Tables
Save good data to main table and bad data to quarantine table

In [0]:
# Write good data to main bronze table
print(f"Writing {validated_df.count()} records to {TABLE_RAW_FULL}...")

validated_df.write \
    .format("Delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE_RAW_FULL)

print("Main table written")

In [0]:
# Write bad data to quarantine table
print(f"Writing {bad_df.count()} records to {TABLE_QUARANTINE_FULL}...")

bad_df.write \
    .format("Delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE_QUARANTINE_FULL)

print("Qurantine table written")

# Verify Tables

Read back and validate the tables and were created ccorrectly 

In [0]:
# Read main table
main_table = spark.read.table(TABLE_RAW_FULL)
quarantine_table = spark.read.table(TABLE_QUARANTINE_FULL)

print("=" * 60)
print("MAIN TABLE SUMMARY")
print("=" * 60)
print(f"Table: {TABLE_RAW_FULL}")
print(f"Records Count: {main_table.count()}")
print("Sample Records")
main_table.select("equipment_id", "sensor_type", "timestamp", "value" , "unit").show(5)


print("\n" + "=" * 60)
print("QUARANTIME TABLE SUMMARY")
print("=" * 60)
print(f"Table {TABLE_QUARANTINE_FULL}")
print(f"Records Count: {quarantine_table.count()}")
print("Quarantine breakdown")
quarantine_table.groupBy("quality_flag").count().show()

## Data Quality Matrix

Calculate and display quality metrics

In [0]:
# Calculate data quality matrics
total_loaded = enriched_df.count()
good_count = main_table.count()
bad_count = quarantine_table.count()

quality_rate = (good_count / total_loaded) * 100

print("=" * 60)
print("DATA QUALITY MATRICS")
print("=" * 60)
print(f"Total records loaded: {total_loaded:,}")
print(f"Good records: {good_count:,}")
print(f"Quarantine records: {bad_count:,}")
print(f"Data quality rate: {quality_rate:.2f}%")
print(f"")
print(f"Tarage quality rate: 95%")
if quality_rate > 95:
    print("Status: Passed")
else:
    print("Status: Failed")

## Sensor Distribute Analysis
Analyze the distribution of sensor types and equipment

In [0]:
from pyspark.sql.functions import count, avg, min, max, stddev

print("Sensor Type Distribution: ")
main_table.groupBy("sensor_type").count().orderBy("count", ascending=False).show()

print("\nTop 10 Equipment by Reading Count: ")
main_table.groupBy("equipment_id").count().orderBy("count", ascending=False).show(10)

print("\nValue Statistics by Sensor Type:")
main_table.groupBy("sensor_type").agg(
    count("*").alias("count"),
    avg("value").alias("avg_value"),
    min("value").alias("min_value"),
    max("value").alias("max_value"),
    stddev("value").alias("stddev_value")
).show()

# Check for duplicates

Idenity duplicates records in the good data

In [0]:
# Check the the duplicates based on key columns
duplicate_check = main_table.groupBy(
    "equipment_id", "sensor_type", "timestamp"
).count().filter("count > 1")

duplicate_count = duplicate_check.count()

print(f"Duplicate records: {duplicate_count}")

if duplicate_count > 0:
    print("Sample Duplicate")
    duplicate_check.show(10)
    print("There will be deduplicated in silver layer")
else:
    print("No duplicate found")


## Summary Report
Final summary of the ingestion process

In [0]:
from datetime import datetime

print("=" * 70)
print("BRONZE SENSOR INGESTION SUMMARY")
print("=" * 70)
print(f"Timestrap: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"")
print(f"SOURCE:")
print(f"File: sensor_readings_10k.csv")
print(f"Location: /Volumes/dev/bronze/sample_data/")
print(f"")
print(f"OUTPUT:")
print(f"Main Table: {TABLE_RAW_FULL}")
print(f"Quarantine Table: {TABLE_QUARANTINE_FULL}")
print(f"")
print(f"METRICS:")
print(f"Total loaded: {total_loaded:,}")
print(f"Good Records: {good_count:,} ({quality_rate:.1f}%)")
print(f"Quarantined: {bad_count:,}")
print(f"Duplicates detected : {duplicate_count}")
print(f"")
print(f"NEXT STEPS:")
print(f"1. Review quarantine table for patterns")
print(f"2. Build silver layer to clean and deduplicate")
print(f"3. Implement automated quality alerts")
print("=" * 70)
print("Bronze ingestion complete!")